#

<div align=center>
<img src="https://uol.unifor.br/acesso/app/autenticacao/assets/img/logos/icon-unifor.svg" width=45 height=45>

<br><br>
<font size=5 color='black'><strong>MBA Ciência de dados:</strong> Estatística descritiva

<strong>Projeto:</strong> Titanic

<strong>Autoria:</strong> Heitor Teixeira

</div>

## <font color=darkblue> 1. Imports e declaração de constantes

### <font color=steelblue> 1.1 Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew, gaussian_kde
import numpy as np
import requests


### <font color=steelblue> 1.2 Constantes

Declarar constantes em uma unica celula facilita a manutenção de notebooks longos.<br>Alguns benefícios:
1. Melhorar a legibilidade do codigo
2. As constantes podem ser esquema de cores para padronizar sempre as mesmas cores para determinadas classes, paths de arquivos e datasets e etc.
3. posso modifica-las apenas aqui e servir para o codigo inteiro


In [2]:
# paths dos arquivos
PATH_TITANIC = 'datasets/titanic.csv'
PATH_BOE = 'datasets/extras/a-millennium-of-macroeconomic-data-for-the-uk.xlsx'

# constantes para  variaveis qualitativas e cabecalho
LABELS_CLASSE = {1: 'Primeira Classe', 2: 'Segunda Classe', 3: 'Terceira Classe'}

LABELS_GENERO = {'male': 'Masculino', 'female': 'Feminino'}

LABELS_SOBREVIVENTE = {0: 'Não Sobreviveu', 1: 'Sobreviveu'}

MAP_COLUMNS = {
    'Survived': 'Sobreviveu',
    'Pclass': 'Classe',
    'Name': 'Nome',
    'Sex': 'Genero',
    'Age': 'Idade',
    'Siblings/Spouses Aboard': 'Irmãos/Cônjuges a Bordo',
    'Parents/Children Aboard': 'Pais/Filhos a Bordo',
    'Fare': 'Tarifa',
}

## <font color=darkblue> 2. Carregamento e Preparação do Dataset

### <font color=steelblue> 2.1 Carregando o dataset original

**Observações**<br>
Esta análise considera `exclusivamente os passageiros do Titanic. Algumas decisões metodológicas foram adotadas para garantir maior clareza e consistência nos resultados:

1. Registros com `Fare = 0` foram excluídos, pois não é possível distinguir se pertencem a passageiros ou tripulantes, e sua presença poderia distorcer métricas e visualizações.
2. As variáveis qualitativas já são carregadas com mapeamento pelos valores definidos nas constantes, padronizando rótulos em tabelas e gráficos sem necessidade de configuração adicional.
3. Por essas razões, pequenas variações nos cálculos de média, mediana, desvio padrão e outras medidas em relação a outras análises do mesmo dataset são esperadas.

In [3]:
# carregando o dataset já renomeado com as constantes definidas na seçao 1.2
titanic_df = pd.read_csv(PATH_TITANIC, header=0, names=MAP_COLUMNS.values())

titanic_df = titanic_df[titanic_df['Tarifa']>0]

titanic_df['Classe'] = titanic_df['Classe'].map(LABELS_CLASSE)
titanic_df['Genero'] = titanic_df['Genero'].map(LABELS_GENERO)
titanic_df['Sobreviveu'] = titanic_df['Sobreviveu'].map(LABELS_SOBREVIVENTE)
titanic_df.tail(1)


,Sobreviveu,Classe,Nome,Genero,Idade,Irmãos/Cônjuges a Bordo,Pais/Filhos a Bordo,Tarifa
886,Não Sobreviveu,Terceira Classe,Mr. Patrick Dooley,Masculino,32.0,0,0,7.75


### <font color=steelblue> 2.2 Feature Engineering

**Observações:***<br>
Em aula, o professor falou da importancia de considerar tanto valores absolutos quanto relativos. No caso do Titanic, como o acidente ocorreu em 1912, as tarifas originais estão completamente defasadas: £1 de 1912 não equivale a £1 hoje. <br><br>
Para preservar a noção de grandeza e permitir uma interpretação mais realista dos preços, adicionei uma coluna com a tarifa corrigida pela inflação. Os índices utilizados foram:

1. `Bank of England` - A Millennium of Macroeconomic Data for the UK (v3.1), aba A47, coluna H. Índice composto de preços do Reino Unido, normalizado para 100 em 2016. Cobre o período de 1912 a 2016.
2. `World Bank` - utilizado para estender a correção de 2016 até os anos mais recentes, complementando a série histórica do BoE.


In [ ]:
boe_df = pd.read_excel(
    PATH_BOE,
    sheet_name="A47. Wages and prices",
)


In [7]:
boe_df.head()

,Back to front page,A47. Wages and Prices 1209-2016,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 72,Unnamed: 73,Unnamed: 74,Unnamed: 75,Unnamed: 76,Unnamed: 77,Unnamed: 78,Unnamed: 79,Unnamed: 80,Unnamed: 81
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Composite Average Weekly Earnings series,NaN,Consumer Price Index (CPI) - preferred measure,CPI inflation,Consumer Price Index (CPI) - original method f...,RPI based index,CPI for Ireland,Spliced Wholesale/producer price index,Implied consumption deflator,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Schumpeter Gilboy -Producer prices,NaN
2,Coverage,England/GB,UK pre-1922 definition (GB+I),GB/UK(GB+NI),GB/UK(GB+NI),GB/UK(GB+NI),GB/UK(GB+NI),Ireland/ROI,GB/UK,UK,...,Ireland,S. Ireland,UK/GB,UK,UK,UK,"Composite Consumer Price index, 1750=100",CPI index 2015=100,1697=100,1701=100
3,Source,NB based on English daily wage rates prior to ...,NaN,"1661-1750, Schumpeter-Gilboy index from Mitche...",NaN,"1661-1750, Schumpeter-Gilboy index from Mitche...","Clark (2009) 1600-1750, ONS (O'Donoghue et al ...",NaN,"Post 1791 data, annual average of the index in...",See GDP(E) component sheets,...,Brunt and Cannon (2014),CSO of Ireland,Source: Feinstein (1991),Capie and Collins (1983),Cost of living index originally from Ministry ...,Source: ONS Retail prices 1919-1990,"Source: O'Donoghue et al (2004), index derived...",Source: ONS website,"Mitchell (1988) Chapter XIV Table 1, page 719","Mitchell (1988) Chapter XIV Table 1, page 719"
4,NaN,£,1913=100,Spliced index 2015=100,annual inflation rate,Spliced index 2015=100,Spliced index 2015=100,Spliced index 2015=100,2010=100,"2013 = 100, based on composite value and volum...",...,Average measure,1914=100,Cost of living 1900=100,Cost of living 1924=100,NaN,"Retail price index, Jan 1987=100",NaN,NaN,NaN,NaN
